In [1]:
# =========================
# CS-340 Project Two Dashboard
# Grazioso Salvare
# Maurion Caldwell
# =========================

from dash import Dash, dcc, html, dash_table
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64


from animal_shelter import AnimalShelter

# JupyterDash.infer_jupyter_proxy_config()

# --------------------
# DB connection (spec: hardcode)
# --------------------
username = "aacuser"
password = "SNHU1234"
db = AnimalShelter(username, password)

# retrieve-all (SPEC STEP 2)
df = pd.DataFrame.from_records(db.read({}))
df.head()
# Dash DataTable can't handle ObjectId well -> remove for DISPLAY only
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

# --------------------
# Logo (SPEC requirement)
# --------------------
import os

image_filename = "logo.png"

if os.path.exists(image_filename):
    encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode("utf-8")
else:
    encoded_image = ""

# --------------------
# App
# --------------------
app = Dash(__name__)

app.layout = html.Div([
    html.Div([
        html.Img(
            src=f"data:image/png;base64,{encoded_image}",
            style={"width": "180px", "display": "block", "margin": "0 auto"}
        ),
        html.H2("Grazioso Salvare Dashboard - Maurion Caldwell", style={"textAlign": "center"})
    ]),
    html.Hr(),

    # Controller (radio buttons)
    dcc.RadioItems(
        id="filter-type",
        options=[
            {"label": "Water Rescue", "value": "water"},
            {"label": "Mountain/Wilderness Rescue", "value": "mountain"},
            {"label": "Disaster/Individual Tracking", "value": "disaster"},
            {"label": "Reset", "value": "reset"},
        ],
        value="reset",
        inline=True,
        style={"textAlign": "center"}
    ),

    html.Hr(),

    # DataTable (SPEC STEP 2: unfiltered view initially)
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),

        # user-friendly features (recommended by spec)
        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[],
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "padding": "6px", "minWidth": "120px", "width": "120px", "maxWidth": "200px"},
    ),

    html.Br(),
    html.Hr(),
    
    html.Div(id="type-graph-id", style={"marginBottom": "20px"}),
        
    html.Div(
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(id="graph-id", style={"width": "45%"}),
            html.Div(id="map-id", style={"width": "55%"}),
        ],
    ),
])

# -------------------------------------------------------
# Callback #1: filter table from MongoDB (spec queries)
# -------------------------------------------------------
@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value"),
)
def update_dashboard(filter_type):
    # SPEC table:
    # Water: Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland
    # Mountain: German Shepherd, Alaskan Malamute, Old English Sheepdog, Siberian Husky, Rottweiler
    # Disaster: Doberman Pinscher, German Shepherd, Golden Retriever, Bloodhound, Rottweiler
    # Plus:
    #   Water & Mountain: sex_upon_outcome = "Intact Female"
    #   Disaster: sex_upon_outcome = "Intact Male"
    #   Water & Mountain: age 26..156 weeks (inclusive-ish)
    #   Disaster: age 20..300 weeks

    if filter_type == "water":
        query = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == "mountain":
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == "disaster":
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")

# -------------------------------------------------------
# Callback #2: graph from current table view
# -------------------------------------------------------
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
)
def update_graphs(viewData):
    if not viewData:
        return html.Div("No data to display (adjust filter or reset).")

    df = pd.DataFrame.from_dict(viewData)

    if "breed" not in df.columns:
        return html.Div("Missing 'breed' field in data.")

    type_counts = df["animal_type"].value_counts().sort_values(ascending=False)

    fig = px.bar(
        x=type_counts.index,
        y=type_counts.values,
        title="Animals by Type",
        labels={"x": "Animal Type", "y": "Count"},
        color=type_counts.index
    )

    fig.update_layout(
        title={"x": 0.5},
        height=400
    )

    return dcc.Graph(figure=fig)
    
#------------------------------------------------------------
# Callback #2 extended: second graph from current table view
#------------------------------------------------------------

@app.callback(
    Output("type-graph-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
)
def update_type_graph(viewData):
    if not viewData:
        return html.Div("No data to display.")

    df = pd.DataFrame.from_dict(viewData)

    if "animal_type" not in df.columns:
        return html.Div("Missing 'animal_type' field.")

    type_counts = df["animal_type"].value_counts()

    fig = px.bar(
        x=type_counts.index,
        y=type_counts.values,
        title="Animals by Type"
    )

    fig.update_layout(
        title={"x": 0.5},
        height=400
    )

    return dcc.Graph(figure=fig)

# -------------------------------------------------------
# Callback #3: map from selected row
# -------------------------------------------------------
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, selected_rows):
    if not viewData:
        return html.Div("No data to map.")

    dff = pd.DataFrame.from_dict(viewData)

    # default to first row if nothing selected
    row = 0
    if selected_rows and len(selected_rows) > 0:
        row = selected_rows[0]

    # field names from AAC dataset
    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return html.Div("Missing location_lat/location_long fields.")

    lat = dff.loc[row, "location_lat"]
    lon = dff.loc[row, "location_long"]

    breed = dff.loc[row, "breed"] if "breed" in dff.columns else "Unknown"
    name = dff.loc[row, "name"] if "name" in dff.columns else "Unknown"

    return dl.Map(
        style={"width": "100%", "height": "500px"},
        center=[30.75, -97.48],
        zoom=10,
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[lat, lon],
                children=[
                    dl.Tooltip(str(breed)),
                    dl.Popup([html.H4("Animal Name"), html.P(str(name))])
                ]
            )
        ]
    )

app.run(debug=False)